In [ ]:
# Standard library imports
import json
import os
import sys
import time
import warnings
from datetime import datetime
from pathlib import Path


# Third-party imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Project imports - Add project root to path
project_root = Path.cwd().parent if 'Notebooks' in str(Path.cwd()) else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.helpers import download_parse_json, parse_json_response
from utils.llm import call_llm, count_tokens, get_model_name
from utils.llm_func import judge_semantic_match, extract_contract_clauses
from utils.clauses_prompts import CLAUSE_TO_TYPE, CLAUSE_DEFINITIONS
# from utils.llm_ollama import call_llm
# call_llm = call_gemini  # Use Gemini for all calls in this notebook
# Configuration
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

# LLM Model Configuration - Two-Model Architecture
EXTRACTION_MODEL = "google/gemini-2.5-flash"        # Model being evaluated (extracts clauses)
JUDGE_MODEL = EXTRACTION_MODEL

# Model for verification (semantic matching)
# openai/gpt-5.2
# openai/gpt-5-nano
# anthropic/claude-sonnet-4.5
# qwen/qwen3-32b
# google/gemma-3-4b-it
# google/gemma-3-27b-it:free
# meta-llama/llama-3.3-8b-instruct

print(f'🤖 Extraction Model: {EXTRACTION_MODEL}')
print(f'⚖️  Judge Model: {JUDGE_MODEL}')


# Batch processing configuration
NUM_CONTRACTS = 120  # Number of contracts to process
# Keep single-pass extraction by default to reduce runtime and cost
NUM_RUNS = 1  # Number of extraction runs per contract
EXTRACTION_CHECKPOINT_INTERVAL = 10  # Save every N contracts

# Folder structure
BASE_DIR = Path('./batch_results')
CHECKPOINT_DIR = BASE_DIR / 'checkpoints'  # Temporary checkpoint files
EXTRACTION_DIR = BASE_DIR / 'extractions'  # Extraction results
JUDGE_DIR = BASE_DIR / 'judge_results'      # Judge evaluation results
SUMMARY_DIR = BASE_DIR / 'summaries'        # Final summary reports
ANALYSIS_DIR = BASE_DIR / 'analysis'        # Error analysis reports

# Create all directories
for dir_path in [BASE_DIR, CHECKPOINT_DIR, EXTRACTION_DIR, JUDGE_DIR, SUMMARY_DIR, ANALYSIS_DIR]:
    dir_path.mkdir(exist_ok=True)

print('✅ Environment configured successfully')
print(f'📁 Project root: {project_root}')
print(f'📁 Output structure:')
print(f'   • Base: {BASE_DIR}')
print(f'   • Checkpoints: {CHECKPOINT_DIR.name}/ (temporary)')
print(f'   • Extractions: {EXTRACTION_DIR.name}/')
print(f'   • Judge Results: {JUDGE_DIR.name}/')
print(f'   • Summaries: {SUMMARY_DIR.name}/')
print(f'   • Analysis: {ANALYSIS_DIR.name}/')
print(f'📊 Processing {NUM_CONTRACTS} contract(s)')

In [ ]:
call_llm("hi",EXTRACTION_MODEL)  # Alias for easier switching of LLM backends

In [ ]:
# Optional: Load extraction data from a previously saved pickle file
# (Skip this cell if you just ran the extraction above - data is already in memory as 'combined_df')

# Uncomment to load from a specific file:
# pkl_file = EXTRACTION_DIR / 'extractions_df_openai-gpt-5.2_20260209_155756.pkl'
# combined_df = pd.read_pickle(pkl_file)
# print(f'📦 Loaded from: {pkl_file.name}')

# Or auto-load the most recent extraction file:
extraction_files = sorted(EXTRACTION_DIR.glob('extractions_df_*.pkl'), key=lambda x: x.stat().st_mtime, reverse=True)
if extraction_files:
    pkl_file = extraction_files[0]
    combined_df = pd.read_pickle(pkl_file)
    print(f'📦 Loaded most recent: {pkl_file.name}')
else:
    print('⚠️  No extraction files found')

# print('✅ Using in-memory data from extraction cell above')
# pkl_file = EXTRACTION_DIR / 'debate_merged_20260413_221353.pkl'
# combined_df = pd.read_pickle(pkl_file)

In [ ]:
# Preview combined dataset
if combined_df is not None:
    print('📋 Sample of combined extraction data:')
    display(combined_df)
else:
    print('⚠️  No data available')

---
## 5. Detection Metrics

Calculate False Acceptance Rate (FAR), False Refusal Rate (FRR), and Detection Accuracy

In [ ]:
# Compute detection metrics
if combined_df is not None:
    # Confusion matrix calculation
    tp = ((combined_df['is_impossible'] == False) & (combined_df['is_impossible_ai'] == False)).sum()
    tn = ((combined_df['is_impossible'] == True) & (combined_df['is_impossible_ai'] == True)).sum()
    fp = ((combined_df['is_impossible'] == True) & (combined_df['is_impossible_ai'] == False)).sum()
    fn = ((combined_df['is_impossible'] == False) & (combined_df['is_impossible_ai'] == True)).sum()
    
    total_negative = (combined_df['is_impossible'] == True).sum()
    total_positive = (combined_df['is_impossible'] == False).sum()
    
    # Metrics calculation
    far = (fp / total_negative * 100) if total_negative > 0 else 0
    frr = (fn / total_positive * 100) if total_positive > 0 else 0
    detection_accuracy = ((tp + tn) / len(combined_df) * 100)
    
    print('='*80)
    print('📊 DETECTION METRICS')
    print('='*80)
    extraction_model = combined_df['extraction_model'].iloc[0] if 'extraction_model' in combined_df.columns else 'unknown'
    print(f'🤖 Extraction Model: {extraction_model}')
    print(f'\n🔍 Confusion Matrix:')
    print(f'   TP (Correct Detection):   {tp:4d}')
    print(f'   TN (Correct Abstention):  {tn:4d}')
    print(f'   FP (False Acceptance):    {fp:4d}')
    print(f'   FN (False Refusal):       {fn:4d}')
    print(f'\n📈 Metrics:')
    print(f'   FAR (False Acceptance):   {far:6.1f}%  (FP / GT-absent)')
    print(f'   FRR (False Refusal):      {frr:6.1f}%  (FN / GT-present)')
    print(f'   Detection Accuracy:       {detection_accuracy:6.1f}%  ((TP+TN) / Total)')
    print('='*80)
    
    # Store for later use
    benchmark_detection = {
        'TP': int(tp), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn),
        'FAR_%': round(far, 1),
        'FRR_%': round(frr, 1),
        'accuracy_%': round(detection_accuracy, 1)
    }
else:

    print('⚠️  No data available for detection metrics')    
    benchmark_detection = None

In [ ]:
# Enhanced Detection Metrics with Run-Level Consistency Analysis

if combined_df is not None:
    print('='*80)
    print('📊 ENHANCED DETECTION METRICS WITH RUN-LEVEL ANALYSIS')
    print('='*80)
    extraction_model = combined_df['extraction_model'].iloc[0] if 'extraction_model' in combined_df.columns else 'unknown'
    print(f'🤖 Extraction Model: {extraction_model}')
    
    # Overall metrics (across all runs)
    tp = ((combined_df['is_impossible'] == False) & (combined_df['is_impossible_ai'] == False)).sum()
    tn = ((combined_df['is_impossible'] == True) & (combined_df['is_impossible_ai'] == True)).sum()
    fp = ((combined_df['is_impossible'] == True) & (combined_df['is_impossible_ai'] == False)).sum()
    fn = ((combined_df['is_impossible'] == False) & (combined_df['is_impossible_ai'] == True)).sum()
    
    total_negative = (combined_df['is_impossible'] == True).sum()
    total_positive = (combined_df['is_impossible'] == False).sum()
    
    far = (fp / total_negative * 100) if total_negative > 0 else 0
    frr = (fn / total_positive * 100) if total_positive > 0 else 0
    detection_accuracy = ((tp + tn) / len(combined_df) * 100)
    
    print(f'\n🎯 OVERALL METRICS (All {NUM_RUNS} runs combined):')
    print(f'   Total Evaluations:  {len(combined_df):4d}')
    print(f'   TP (Correct Detection):   {tp:4d}')
    print(f'   TN (Correct Abstention):  {tn:4d}')
    print(f'   FP (False Acceptance):    {fp:4d}')
    print(f'   FN (False Refusal):       {fn:4d}')
    print(f'\n   FAR (False Acceptance):   {far:6.1f}%')
    print(f'   FRR (False Refusal):      {frr:6.1f}%')
    print(f'   Detection Accuracy:       {detection_accuracy:6.1f}%')
    
    # Per-run breakdown
    print(f'\n📋 RUN-LEVEL BREAKDOWN:')
    print(f'{"Run":<6} {"Total":<7} {"TP":<5} {"TN":<5} {"FP":<5} {"FN":<5} {"FAR%":<7} {"FRR%":<7} {"Acc%":<7}')
    print('-' * 70)
    
    run_metrics = []
    for run_id in sorted(combined_df['run_id'].unique()):
        run_df = combined_df[combined_df['run_id'] == run_id]
        
        run_tp = ((run_df['is_impossible'] == False) & (run_df['is_impossible_ai'] == False)).sum()
        run_tn = ((run_df['is_impossible'] == True) & (run_df['is_impossible_ai'] == True)).sum()
        run_fp = ((run_df['is_impossible'] == True) & (run_df['is_impossible_ai'] == False)).sum()
        run_fn = ((run_df['is_impossible'] == False) & (run_df['is_impossible_ai'] == True)).sum()
        
        run_neg = (run_df['is_impossible'] == True).sum()
        run_pos = (run_df['is_impossible'] == False).sum()
        
        run_far = (run_fp / run_neg * 100) if run_neg > 0 else 0
        run_frr = (run_fn / run_pos * 100) if run_pos > 0 else 0
        run_acc = ((run_tp + run_tn) / len(run_df) * 100)
        
        print(f'{run_id:<6} {len(run_df):<7} {run_tp:<5} {run_tn:<5} {run_fp:<5} {run_fn:<5} {run_far:<7.1f} {run_frr:<7.1f} {run_acc:<7.1f}')
        
        run_metrics.append({
            'run_id': run_id,
            'far_%': run_far,
            'frr_%': run_frr,
            'accuracy_%': run_acc,
            'fp': int(run_fp),
            'fn': int(run_fn)
        })
    
    # Consistency analysis
    print(f'\n🔍 CONSISTENCY ANALYSIS:')
    far_std = np.std([m['far_%'] for m in run_metrics])
    frr_std = np.std([m['frr_%'] for m in run_metrics])
    acc_std = np.std([m['accuracy_%'] for m in run_metrics])
    
    print(f'   FAR Variability:  σ = {far_std:.2f}%  (lower is better)')
    print(f'   FRR Variability:  σ = {frr_std:.2f}%  (lower is better)')
    print(f'   Accuracy Std Dev: σ = {acc_std:.2f}%  (lower is better)')
    
    # Identify inconsistent clauses
    clause_consistency = combined_df.groupby('clause_name').agg({
        'is_impossible_ai': lambda x: x.nunique()  # Count unique predictions per clause
    }).rename(columns={'is_impossible_ai': 'prediction_variants'})
    
    inconsistent_clauses = clause_consistency[clause_consistency['prediction_variants'] > 1]
    
    if len(inconsistent_clauses) > 0:
        print(f'\n⚠️  INCONSISTENT PREDICTIONS DETECTED:')
        print(f'   {len(inconsistent_clauses)} clauses had different predictions across runs')
        print(f'\n   Top 5 Inconsistent Clauses:')
        for clause_name in inconsistent_clauses.head(5).index:
            clause_runs = combined_df[combined_df['clause_name'] == clause_name]
            detected = (~clause_runs['is_impossible_ai']).sum()
            abstained = (clause_runs['is_impossible_ai']).sum()
            gt_status = 'Present' if not clause_runs['is_impossible'].iloc[0] else 'Absent'
            print(f'      • {clause_name[:50]}...')
            print(f'        GT: {gt_status} | Detected: {detected}/{NUM_RUNS} | Abstained: {abstained}/{NUM_RUNS}')
    else:
        print(f'\n✅ PERFECT CONSISTENCY: All clauses received identical predictions across {NUM_RUNS} runs')
    
    # Store enhanced metrics
    benchmark_detection = {
        'TP': int(tp), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn),
        'FAR_%': round(far, 1),
        'FRR_%': round(frr, 1),
        'accuracy_%': round(detection_accuracy, 1),
        'per_run': run_metrics,
        'consistency': {
            'far_std': round(far_std, 2),
            'frr_std': round(frr_std, 2),
            'acc_std': round(acc_std, 2),
            'inconsistent_clauses': len(inconsistent_clauses)
        }
    }
    
    print('='*80)
else:

    print('⚠️  No data available for detection metrics')    
    benchmark_detection = None

---
## 6. LLM Judge Evaluation

Evaluate semantic equivalence between AI-generated and ground truth answers

In [ ]:
# Run LLM judge with checkpoint saving and resume capability
if combined_df is not None:
    # Configuration
    CHECKPOINT_INTERVAL = 500  # Save every N clauses (adjust based on your needs)
    
    # Filter to clauses that need judging
    clauses_to_judge = combined_df[
        (combined_df['is_impossible'] == False) & 
        (combined_df['is_impossible_ai'] == False)
    ].copy()
    
    # Add unique identifier for resume logic
    clauses_to_judge['judge_key'] = (
        clauses_to_judge['title'] + '||' + 
        clauses_to_judge['clause_name'] + '||' + 
        clauses_to_judge['run_id'].astype(str)
    )
    
    print('='*80)
    print('⚖️  LLM JUDGE EVALUATION (WITH CHECKPOINT RESUME)')
    print('='*80)
    print(f'Judge Model: {JUDGE_MODEL}')
    print(f'Clauses to evaluate: {len(clauses_to_judge):,}')
    print(f'Checkpoint interval: Every {CHECKPOINT_INTERVAL} clauses')
    print(f'Estimated time: {len(clauses_to_judge) * 0.5 / 60:.1f} minutes\n')
    
    # Setup checkpoint file paths (fixed filename for resume)
    checkpoint_pkl = CHECKPOINT_DIR / 'judge_checkpoint.pkl'
    
    # Generate timestamp and model names for final output files
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    extraction_model_name = combined_df['extraction_model'].iloc[0].replace('/', '-')
    from utils.llm import get_model_name
    judge_model_name = get_model_name(JUDGE_MODEL).replace('/', '-')
    judge_pkl = JUDGE_DIR / f'judge_results_{extraction_model_name}_judged-by_{judge_model_name}_{timestamp}.pkl'
    judge_csv = JUDGE_DIR / f'judge_results_{extraction_model_name}_judged-by_{judge_model_name}_{timestamp}.csv'
    
    # Resume logic: Load existing results if checkpoint exists
    if checkpoint_pkl.exists():
        print(f'📂 Found checkpoint file: {checkpoint_pkl.name}')
        results_df = pd.read_pickle(checkpoint_pkl)
        
        # Create set of already-processed keys
        processed_keys = set(results_df['judge_key'])
        
        # Filter out already processed clauses
        clauses_to_judge = clauses_to_judge[~clauses_to_judge['judge_key'].isin(processed_keys)]
        
        print(f'✅ Resumed from checkpoint')
        print(f'   Already processed: {len(processed_keys):,}')
        print(f'   Remaining: {len(clauses_to_judge):,}')
        print(f'   Progress: {len(processed_keys)/(len(processed_keys) + len(clauses_to_judge))*100:.1f}%\n')
    else:
        print('📝 Starting fresh evaluation (no checkpoint found)\n')
        results_df = pd.DataFrame()
        processed_keys = set()
    
    failed_clauses = []
    clauses_processed_since_checkpoint = 0
    judge_rows = []
    
    for idx, row in clauses_to_judge.iterrows():
        # Retry logic for robustness
        max_retries = 3
        retry_delay = 2
        
        for attempt in range(max_retries):
            try:
                is_match, reason, mismatch_type, judge_model = judge_semantic_match(
                    row['clause_name'],
                    str(row['answer_ai']),
                    str(row['answers']),
                    model=JUDGE_MODEL
                )
                
                # Create judge result row by copying all extraction columns and adding judge columns
                judge_row = row.to_dict()
                judge_row.update({
                    'llm_match': is_match,
                    'judge_reason': reason,
                    'judge_mismatch_type': mismatch_type,
                    'claim_status': 'SUPPORTED' if is_match else 'CONTRADICTED',
                    'judge_model': judge_model
                })
                judge_rows.append(judge_row)
                
                clauses_processed_since_checkpoint += 1
                total_processed = len(results_df) + len(judge_rows)
                
                # Progress indicator
                if total_processed % 10 == 0 or not is_match:
                    emoji = '✅' if is_match else '❌'
                    print(f'{emoji} [{total_processed:,}] {row["clause_name"][:60]}')
                
                # CHECKPOINT SAVE: Save every N clauses
                if clauses_processed_since_checkpoint >= CHECKPOINT_INTERVAL:
                    # Append new results to existing dataframe
                    new_results = pd.DataFrame(judge_rows)
                    results_df = pd.concat([results_df, new_results], ignore_index=True)
                    judge_rows = []  # Clear buffer
                    
                    results_df.to_pickle(checkpoint_pkl)
                    clauses_processed_since_checkpoint = 0
                    print(f'   💾 Checkpoint saved: {total_processed:,} clauses processed')
                
                break
                
            except Exception as e:
                if attempt < max_retries - 1:
                    print(f'   ⚠️  Retry {attempt + 1}/{max_retries}: {str(e)[:60]}...')
                    time.sleep(retry_delay * (attempt + 1))
                else:
                    print(f'   ❌ Failed: {row["clause_name"]}')
                    failed_clauses.append({'clause': row['clause_name'], 'error': str(e)})
                    
                    # Add as contradicted on failure
                    from utils.llm import get_model_name
                    judge_row = row.to_dict()
                    judge_row.update({
                        'llm_match': False,
                        'judge_reason': f'Evaluation failed: {str(e)[:100]}',
                        'judge_mismatch_type': 'evaluation_error',
                        'claim_status': 'CONTRADICTED',
                        'judge_model': get_model_name(JUDGE_MODEL)
                    })
                    judge_rows.append(judge_row)
                    clauses_processed_since_checkpoint += 1
    
    # Final save - append any remaining results
    if len(judge_rows) > 0:
        new_results = pd.DataFrame(judge_rows)
        results_df = pd.concat([results_df, new_results], ignore_index=True)
    
    results_df.to_pickle(judge_pkl)
    results_df.to_csv(judge_csv, index=False)
    
    # Clean up checkpoint file after successful completion
    if checkpoint_pkl.exists():
        checkpoint_pkl.unlink()
        print(f'\n🗑️  Checkpoint file removed (evaluation complete)')
    
    print(f'\n✅ Evaluation complete!')
    print(f'   Total processed: {len(results_df):,}')
    print(f'   Success: {len(results_df) - len(failed_clauses)}/{len(results_df)}')
    if failed_clauses:
        print(f'   Failed: {len(failed_clauses)}')
    
    print(f'\n💾 Saved final judge results:')
    print(f'   • {judge_pkl.name}')
    print(f'   • {judge_csv.name} (backup)')
    print('='*80)

else:
    print('⚠️  No data available for judging')
    results_df = None

In [ ]:
results_df

---
## 7. Content Quality Metrics

Calculate precision, hallucination rate, and judge equivalence metrics

In [ ]:
# Compute content quality metrics from judge results
if results_df is not None and combined_df is not None:
    supported = (results_df['llm_match'] == True).sum()
    contradicted = (results_df['llm_match'] == False).sum()
    gt_present_count = (~combined_df['is_impossible']).sum()
    
    # Calculate metrics
    precision = supported / (supported + contradicted) if (supported + contradicted) > 0 else 0
    hallucination = contradicted / (supported + contradicted) if (supported + contradicted) > 0 else 0
    judge_equiv = supported / gt_present_count if gt_present_count > 0 else 0
    
    print('='*80)
    print('📊 CONTENT QUALITY METRICS')
    print('='*80)
    extraction_model = results_df['extraction_model'].iloc[0] if 'extraction_model' in results_df.columns else 'unknown'
    judge_model = results_df['judge_model'].iloc[0] if 'judge_model' in results_df.columns else 'unknown'
    print(f'🤖 Extraction Model: {extraction_model}')
    print(f'⚖️  Judge Model: {judge_model}')
    print(f'\n🔍 Judge Results:')
    print(f'   SUPPORTED:      {supported:4d}  (semantically correct)')
    print(f'   CONTRADICTED:   {contradicted:4d}  (semantically incorrect)')
    print(f'   Total Judged:   {supported + contradicted:4d}')
    print(f'\n📈 Metrics:')
    print(f'   PrecisionWhenAnswered:           {precision*100:6.1f}%')
    print(f'      → Quality when AI provides answer')
    print(f'      → SUPPORTED / (SUPPORTED + CONTRADICTED)')
    print(f'\n   HallucinationWhenAnswered:       {hallucination*100:6.1f}%')
    print(f'      → Error rate when AI provides content')
    print(f'      → CONTRADICTED / (SUPPORTED + CONTRADICTED)')
    print(f'\n   JudgeEquivalence@GT-present:     {judge_equiv*100:6.1f}%')
    print(f'      → End-to-end success on answerable clauses')
    print(f'      → SUPPORTED / GT-present count')
    print('='*80)
    
    benchmark_content = {
        'supported': int(supported),
        'contradicted': int(contradicted),
        'precision_%': round(precision * 100, 1),
        'hallucination_%': round(hallucination * 100, 1),
        'judge_equiv@gt_%': round(judge_equiv * 100, 1)

    }    

else:    
    print('⚠️  No data available for content metrics')
    benchmark_content = None

---
## 9. Error Analysis

Detailed analysis of mismatches and failure patterns

In [ ]:
# Detailed mismatch analysis
if results_df is not None:
    mismatch_df = results_df[results_df['llm_match'] == False].copy()
    
    print('='*80)
    print('🔍 ERROR ANALYSIS')
    print('='*80)
    extraction_model = mismatch_df['extraction_model'].iloc[0] if 'extraction_model' in mismatch_df.columns and len(mismatch_df) > 0 else 'unknown'
    judge_model = mismatch_df['judge_model'].iloc[0] if 'judge_model' in mismatch_df.columns and len(mismatch_df) > 0 else 'unknown'
    print(f'🤖 Extraction Model: {extraction_model}')
    print(f'⚖️  Judge Model: {judge_model}')
    print(f'Total mismatches: {len(mismatch_df)}\n')
    
    if len(mismatch_df) > 0:
        # Show sample mismatches
        print('Sample Mismatches (first 3):\n')
        for i, (idx, row) in enumerate(mismatch_df.head(3).iterrows(), 1):
            print(f'{"-"*80}')
            print(f'#{i}')
            print(f'Contract: {row["title"][:60]}...')
            print(f'Clause: {row["clause_name"]}')
            print(f'Type: {row["clause_type"]}')
            print(f'Mismatch Type: {row["judge_mismatch_type"]}')
            print(f'\n🤖 AI Answer: {str(row["debate_answer"])[:150]}...')
            print(f'\n✅ Ground Truth: {str(row["answers"])[:150]}...')
            print(f'\n💬 Reason: {row["judge_reason"][:200]}')
            print(f'{"-"*80}\n')
        
        # Mismatch type statistics
        print('📊 Mismatch Type Distribution:')
        for mtype, count in mismatch_df['judge_mismatch_type'].value_counts().items():
            pct = count / len(mismatch_df) * 100
            print(f'   {mtype:30s}: {count:3d} ({pct:5.1f}%)')
        
        # Mismatch by clause type
        print('\n📊 Mismatches by Clause Type:')
        for ctype, count in mismatch_df.groupby('clause_type').size().items():
            type_total = len(results_df[results_df['clause_type'] == ctype])
            pct = count / type_total * 100 if type_total > 0 else 0
            print(f'   {ctype:30s}: {count:3d}/{type_total:3d} ({pct:5.1f}%)')
        
        # Save mismatch report
        timestamp = batch_start_time.strftime("%Y%m%d_%H%M%S")
        extraction_model_name = mismatch_df['extraction_model'].iloc[0].replace('/', '-')
        judge_model_name = mismatch_df['judge_model'].iloc[0].replace('/', '-')
        mismatch_file = ANALYSIS_DIR / f'mismatch_analysis_{extraction_model_name}_judged-by_{judge_model_name}_{timestamp}.csv'
        mismatch_df.to_csv(mismatch_file, index=False)
        print(f'\n💾 Saved mismatch report: {mismatch_file.name}')
    else:
        print('✅ No mismatches detected!')

        print('⚠️  No results data available')


    print('='*80)    

---
## 10. Final Summary Report

Generate comprehensive summary with all metrics and export results

In [ ]:
# Generate final summary report with typed breakdown
if all([combined_df is not None, results_df is not None, 
    benchmark_detection is not None, benchmark_content is not None]):
    
    # Compute comprehensive typed breakdown using extraction df (combined_df) and judge results
    typed_breakdown = []
    
    # Get all unique clause types from combined_df (has all clauses)
    for clause_type in sorted(combined_df['clause_type'].unique()):
        # Get extraction data for this type (all clauses)
        type_extraction = combined_df[combined_df['clause_type'] == clause_type]
        
        # Detection metrics for this type (from combined_df)
        type_tp = ((type_extraction['is_impossible'] == False) & 
                   (type_extraction['is_impossible_ai'] == False)).sum()
        type_tn = ((type_extraction['is_impossible'] == True) & 
                   (type_extraction['is_impossible_ai'] == True)).sum()
        type_fp = ((type_extraction['is_impossible'] == True) & 
                   (type_extraction['is_impossible_ai'] == False)).sum()
        type_fn = ((type_extraction['is_impossible'] == False) & 
                   (type_extraction['is_impossible_ai'] == True)).sum()
        
        type_total_neg = (type_extraction['is_impossible'] == True).sum()
        type_total_pos = (type_extraction['is_impossible'] == False).sum()
        
        type_far = (type_fp / type_total_neg * 100) if type_total_neg > 0 else None
        type_frr = (type_fn / type_total_pos * 100) if type_total_pos > 0 else None
        type_accuracy = ((type_tp + type_tn) / len(type_extraction) * 100) if len(type_extraction) > 0 else None
        
        # Content metrics for this type (from judge results)
        type_judge_results = results_df[results_df['clause_type'] == clause_type]
        type_supported = (type_judge_results['llm_match'] == True).sum()
        type_contradicted = (type_judge_results['llm_match'] == False).sum()
        type_total_judged = len(type_judge_results)
        
        type_precision = (type_supported / type_total_judged * 100) if type_total_judged > 0 else None
        type_hallucination = (type_contradicted / type_total_judged * 100) if type_total_judged > 0 else None
        
        # Judge equivalence (SUPPORTED / GT-present)
        type_judge_equiv = (type_supported / type_total_pos * 100) if type_total_pos > 0 else None
        
        typed_breakdown.append({
            'clause_type': clause_type,
            'total_clauses': len(type_extraction),
            'gt_present': int(type_total_pos),
            'gt_absent': int(type_total_neg),
            # Detection metrics
            'TP': int(type_tp),
            'TN': int(type_tn),
            'FP': int(type_fp),
            'FN': int(type_fn),
            'far_%': round(type_far, 1) if type_far is not None else None,
            'frr_%': round(type_frr, 1) if type_frr is not None else None,
            'detection_acc_%': round(type_accuracy, 1) if type_accuracy is not None else None,
            # Content metrics
            'total_judged': int(type_total_judged),
            'supported': int(type_supported),
            'contradicted': int(type_contradicted),
            'precision_%': round(type_precision, 1) if type_precision is not None else None,
            'hallucination_%': round(type_hallucination, 1) if type_hallucination is not None else None,
            'judge_equiv@gt_%': round(type_judge_equiv, 1) if type_judge_equiv is not None else None
        })
    
    typed_breakdown_df = pd.DataFrame(typed_breakdown)
    
    print('='*80)
    print('📊 LegalHalluLens EVALUATION SUMMARY')
    print('='*80)
    extraction_model = results_df['extraction_model'].iloc[0] if 'extraction_model' in results_df.columns else 'unknown'
    judge_model = results_df['judge_model'].iloc[0] if 'judge_model' in results_df.columns else 'unknown'
    print(f'🤖 Extraction Model: {extraction_model}')
    print(f'⚖️  Judge Model: {judge_model}')
    
    # Overall metrics
    print('\n🎯 OVERALL PERFORMANCE')
    print(f'   FAR (False Acceptance):          {benchmark_detection["FAR_%"]:6.1f}%')
    print(f'   FRR (False Refusal):             {benchmark_detection["FRR_%"]:6.1f}%')
    print(f'   Detection Accuracy:              {benchmark_detection["accuracy_%"]:6.1f}%')
    print(f'   PrecisionWhenAnswered:           {benchmark_content["precision_%"]:6.1f}%')
    print(f'   HallucinationWhenAnswered:       {benchmark_content["hallucination_%"]:6.1f}%')
    print(f'   JudgeEquivalence@GT-present:     {benchmark_content["judge_equiv@gt_%"]:6.1f}%')
    
    # Split typed breakdown into 2 separate tables for clarity
    
    # Table 1: Detection Metrics by Clause Type
    detection_by_type = typed_breakdown_df[[
        'clause_type', 'total_clauses', 'gt_present', 'gt_absent',
        'TP', 'TN', 'FP', 'FN', 'far_%', 'frr_%', 'detection_acc_%'
    ]].copy()
    
    # Rename columns for better readability
    detection_by_type.columns = [
        'Clause Type', 'Total', 'GT Present', 'GT Absent',
        'TP', 'TN', 'FP', 'FN', 'FAR %', 'FRR %', 'Detection Acc %'
    ]
    
    # Table 2: Content Quality Metrics by Clause Type
    content_by_type = typed_breakdown_df[[
        'clause_type', 'total_judged', 'supported', 'contradicted',
        'precision_%', 'hallucination_%', 'judge_equiv@gt_%'
    ]].copy()
    
    # Rename columns for better readability
    content_by_type.columns = [
        'Clause Type', 'Judged', 'Supported', 'Contradicted',
        'Precision %', 'Hallucination %', 'Judge Equiv %'
    ]
    
    print('\n📋 TABLE 1: DETECTION METRICS BY CLAUSE TYPE')
    print('─' * 120)
    # Format with proper alignment and spacing
    pd.set_option('display.width', 120)
    pd.set_option('display.max_columns', None)
    pd.set_option('display.colheader_justify', 'center')
    print(detection_by_type.to_string(index=False, col_space=12, justify='center'))
    
    print('\n\n📋 TABLE 2: CONTENT QUALITY METRICS BY CLAUSE TYPE')
    print('─' * 100)
    pd.set_option('display.width', 100)
    print(content_by_type.to_string(index=False, col_space=15, justify='center'))
    
    # Processing statistics
    print(f'\n📦 PROCESSING STATISTICS')
    print(f'   Contracts Processed:   {len(successful_extractions)}')
    print(f'   Total Clauses:         {len(results_df):,}')
    print(f'   Judge Evaluations:     {len(results_df):,}')
    print(f'   Processing Time:       {batch_duration:.1f}s ({batch_duration/60:.1f} min)')
    
    # Create summary data structure
    summary_data = {
        'timestamp': batch_start_time.strftime("%Y-%m-%d %H:%M:%S"),
        'models': {
            'extraction_model': extraction_model,
            'judge_model': judge_model
        },
        'config': {
            'num_contracts': len(successful_extractions),
            'num_runs': NUM_RUNS,
            'total_clauses': len(results_df),
            'processing_time_s': round(batch_duration, 1)
        },
        'overall_metrics': {
            'FAR_%': float(benchmark_detection["FAR_%"]),
            'FRR_%': float(benchmark_detection["FRR_%"]),
            'accuracy_%': float(benchmark_detection["accuracy_%"]),
            'precision_%': float(benchmark_content["precision_%"]),
            'hallucination_%': float(benchmark_content["hallucination_%"]),
            'judge_equiv@gt_%': float(benchmark_content["judge_equiv@gt_%"])
        },
        'confusion_matrix': {
            'TP': int(benchmark_detection['TP']),
            'TN': int(benchmark_detection['TN']),
            'FP': int(benchmark_detection['FP']),
            'FN': int(benchmark_detection['FN'])
        },
        'consistency': benchmark_detection.get('consistency', {}),
        'typed_breakdown': typed_breakdown_df.to_dict('records')
    }
    
    timestamp = batch_start_time.strftime("%Y%m%d_%H%M%S")
    extraction_model_name = extraction_model.replace('/', '-')
    judge_model_name = judge_model.replace('/', '-')
    summary_file = SUMMARY_DIR / f'final_summary_{extraction_model_name}_judged-by_{judge_model_name}_{timestamp}.json'
    
    with open(summary_file, 'w') as f:
        json.dump(summary_data, f, indent=2)
    
    print(f'\n💾 Saved final summary: {summary_file.name}')
    print(f'📂 All results saved to: {BASE_DIR}')
    print('='*80)

else:
    typed_breakdown_df = None

    print('⚠️  Insufficient data for final summary')

In [ ]:
from datetime import datetime
from pathlib import Path
import json
import numpy as np
import pandas as pd

project_root = Path.cwd().parent if "Notebooks" in str(Path.cwd()) else Path.cwd()
COMPARISON_ROOT = project_root / "Notebooks"
OVERLAP_KEYS = ["title", "clause_name", "run_id"]
COMPARISON_OUTPUT_DIR = COMPARISON_ROOT / "batch_results" / "comparisons"
COMPARISON_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BASELINE_EXTRACTION_PATH = COMPARISON_ROOT / "batch_results" / "extractions" / "combined_extractions_baseline_gemini-2.5-flash_20260319_220647.csv"
BASELINE_JUDGE_PATH = COMPARISON_ROOT / "batch_results" / "judge_results" / "judge_results_openai-gpt-5.2_judged-by_google-gemini-2.5-flash_20260209_191508.pkl"
DEBATE_EXTRACTION_PATH = COMPARISON_ROOT / "debate_results" / "extractions" / "debate_extractions_ollama-qwen3:8b_20260320_110958.csv"
DEBATE_JUDGE_PATH = COMPARISON_ROOT / "debate_results" / "judge_results" / "debate_judge_ollama-qwen3:8b_by_ollama-qwen3:8b_20260320_115418.csv"

def load_artifact(path):
    if path.suffix == ".csv":
        return pd.read_csv(path)
    if path.suffix == ".pkl":
        return pd.read_pickle(path)
    raise ValueError(f"Unsupported artifact type: {path}")

baseline_extractions_df = load_artifact(BASELINE_EXTRACTION_PATH)
baseline_judge_df = load_artifact(BASELINE_JUDGE_PATH)
debate_extractions_df = load_artifact(DEBATE_EXTRACTION_PATH)
debate_judge_df = load_artifact(DEBATE_JUDGE_PATH)

print(f"Baseline extraction: {BASELINE_EXTRACTION_PATH.name} ({len(baseline_extractions_df)} rows)")
print(f"Baseline judge     : {BASELINE_JUDGE_PATH.name} ({len(baseline_judge_df)} rows)")
print(f"Debate extraction  : {DEBATE_EXTRACTION_PATH.name} ({len(debate_extractions_df)} rows)")
print(f"Debate judge       : {DEBATE_JUDGE_PATH.name} ({len(debate_judge_df)} rows)")
print(f"Overlap keys       : {OVERLAP_KEYS}")

def align_on_overlap(left_df, right_df, keys):
    left_keys = left_df[keys].drop_duplicates()
    right_keys = right_df[keys].drop_duplicates()
    shared_keys = left_keys.merge(right_keys, on=keys, how="inner")
    left_overlap = left_df.merge(shared_keys, on=keys, how="inner").copy()
    right_overlap = right_df.merge(shared_keys, on=keys, how="inner").copy()
    return shared_keys, left_overlap, right_overlap

def compute_detection_metrics(df):
    counts = df["detection_eval"].value_counts()
    tp = int(counts.get("TP", 0))
    fp = int(counts.get("FP", 0))
    fn = int(counts.get("FN", 0))
    tn = int(counts.get("TN", 0))
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    accuracy = (tp + tn) / (tp + fp + fn + tn) if (tp + fp + fn + tn) else 0.0
    far = fp / (fp + tn) if (fp + tn) else 0.0
    frr = fn / (tp + fn) if (tp + fn) else 0.0
    return {
        "rows": int(len(df)),
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "accuracy": accuracy,
        "far": far,
        "frr": frr,
    }

def compute_content_metrics(df):
    status_counts = df["claim_status"].value_counts(dropna=False)
    total = int(len(df))
    supported = int(status_counts.get("SUPPORTED", 0))
    contradicted = int(status_counts.get("CONTRADICTED", 0))
    no_relation = int(status_counts.get("NO_RELATION", 0))
    hallucination_rate = (contradicted + no_relation) / total if total else 0.0
    support_rate = supported / total if total else 0.0
    judge_match_rate = float(df["llm_match"].fillna(False).astype(bool).mean()) if total else 0.0
    return {
        "rows": total,
        "supported": supported,
        "contradicted": contradicted,
        "no_relation": no_relation,
        "support_rate": support_rate,
        "hallucination_rate": hallucination_rate,
        "judge_match_rate": judge_match_rate,
    }


In [ ]:
shared_detection_keys, baseline_detection_overlap, debate_detection_overlap = align_on_overlap(
    baseline_extractions_df,
    debate_extractions_df,
    OVERLAP_KEYS,
)

if shared_detection_keys.empty:
    raise ValueError("No overlapping detection rows found between baseline and debate artifacts.")

baseline_detection_overlap["predicted_present"] = ~baseline_detection_overlap["is_impossible_ai"].fillna(True)
baseline_detection_overlap["actual_present"] = ~baseline_detection_overlap["is_impossible"].fillna(True)
baseline_detection_overlap["detection_eval"] = np.select(
    [
        baseline_detection_overlap["predicted_present"] & baseline_detection_overlap["actual_present"],
        baseline_detection_overlap["predicted_present"] & ~baseline_detection_overlap["actual_present"],
        ~baseline_detection_overlap["predicted_present"] & baseline_detection_overlap["actual_present"],
    ],
    ["TP", "FP", "FN"],
    default="TN",
)

debate_detection_overlap["predicted_present"] = ~debate_detection_overlap["is_impossible_ai"].fillna(True)
debate_detection_overlap["actual_present"] = ~debate_detection_overlap["is_impossible"].fillna(True)
debate_detection_overlap["detection_eval"] = np.select(
    [
        debate_detection_overlap["predicted_present"] & debate_detection_overlap["actual_present"],
        debate_detection_overlap["predicted_present"] & ~debate_detection_overlap["actual_present"],
        ~debate_detection_overlap["predicted_present"] & debate_detection_overlap["actual_present"],
    ],
    ["TP", "FP", "FN"],
    default="TN",
)

baseline_detection_metrics = compute_detection_metrics(baseline_detection_overlap)
debate_detection_metrics = compute_detection_metrics(debate_detection_overlap)
detection_comparison_df = pd.DataFrame([
    {
        "metric": metric,
        "baseline": baseline_detection_metrics[metric],
        "debate": debate_detection_metrics[metric],
        "delta": debate_detection_metrics[metric] - baseline_detection_metrics[metric],
    }
    for metric in ["rows", "TP", "FP", "FN", "TN", "precision", "recall", "f1", "accuracy", "far", "frr"]
])

shared_judge_keys, baseline_judge_overlap, debate_judge_overlap = align_on_overlap(
    baseline_judge_df,
    debate_judge_df,
    OVERLAP_KEYS,
)

if shared_judge_keys.empty:
    raise ValueError("No overlapping judge rows found between baseline and debate artifacts.")

baseline_content_metrics = compute_content_metrics(baseline_judge_overlap)
debate_content_metrics = compute_content_metrics(debate_judge_overlap)
content_comparison_df = pd.DataFrame([
    {
        "metric": metric,
        "baseline": baseline_content_metrics[metric],
        "debate": debate_content_metrics[metric],
        "delta": debate_content_metrics[metric] - baseline_content_metrics[metric],
    }
    for metric in ["rows", "supported", "contradicted", "no_relation", "support_rate", "hallucination_rate", "judge_match_rate"]
])

typed_detection_df = pd.concat([
    baseline_detection_overlap.groupby("clause_type")["detection_eval"].value_counts().unstack(fill_value=0).assign(system="baseline").reset_index(),
    debate_detection_overlap.groupby("clause_type")["detection_eval"].value_counts().unstack(fill_value=0).assign(system="debate").reset_index(),
], ignore_index=True)

report = {
    "generated_at": datetime.now().isoformat(timespec="seconds"),
    "overlap_keys": OVERLAP_KEYS,
    "baseline_extraction_path": str(BASELINE_EXTRACTION_PATH),
    "baseline_judge_path": str(BASELINE_JUDGE_PATH),
    "debate_extraction_path": str(DEBATE_EXTRACTION_PATH),
    "debate_judge_path": str(DEBATE_JUDGE_PATH),
    "shared_detection_rows": int(len(shared_detection_keys)),
    "shared_judge_rows": int(len(shared_judge_keys)),
    "detection_metrics": detection_comparison_df.to_dict(orient="records"),
    "content_metrics": content_comparison_df.to_dict(orient="records"),
}

report_path = COMPARISON_OUTPUT_DIR / f"baseline_vs_debate_overlap_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(report_path, "w") as f:
    json.dump(report, f, indent=2)

print(f"Shared detection rows: {len(shared_detection_keys)}")
print(f"Shared judge rows    : {len(shared_judge_keys)}")
print(f"Saved comparison     : {report_path}")
print()
print("Detection comparison")
display(detection_comparison_df)
print("Content comparison")
display(content_comparison_df)
print("Typed detection counts")
display(typed_detection_df.sort_values(["clause_type", "system"]).reset_index(drop=True))
